In [2]:
# ============================================================
# URBAN MOBILITY INTELLIGENCE SYSTEM
# Notebook 01 — Synthetic Dataset Generation
# Simulated City: Bangalore | Company: Rapido
# Author: [Your Name]
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os
import warnings
warnings.filterwarnings('ignore')

# Reproducibility — every run produces identical data
np.random.seed(42)
random.seed(42)

# ── Paths ────────────────────────────────────────────────────
RAW_DATA_PATH = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\Data\Raw"

# ── Simulation Parameters ────────────────────────────────────
CITY                = "Bangalore"
TOTAL_RIDES         = 500_000
TOTAL_DRIVERS       = 2_000
TOTAL_ZONES         = 200
START_DATE          = datetime(2024, 1, 1)
END_DATE            = datetime(2024, 12, 31)

print("=" * 60)
print("  URBAN MOBILITY INTELLIGENCE SYSTEM")
print(f"  City             : {CITY}")
print(f"  Simulation Start : {START_DATE.date()}")
print(f"  Simulation End   : {END_DATE.date()}")
print(f"  Total Rides      : {TOTAL_RIDES:,}")
print(f"  Total Drivers    : {TOTAL_DRIVERS:,}")
print(f"  Total Zones      : {TOTAL_ZONES}")
print("=" * 60)
print("\n✓ Block 1 complete - Configuration loaded.")

  URBAN MOBILITY INTELLIGENCE SYSTEM
  City             : Bangalore
  Simulation Start : 2024-01-01
  Simulation End   : 2024-12-31
  Total Rides      : 500,000
  Total Drivers    : 2,000
  Total Zones      : 200

✓ Block 1 complete - Configuration loaded.


In [4]:
# ============================================================
# BLOCK 2 - Zone Dataset Generation (CORRECTED)
# 200 Bangalore zones with deduplication handling
# ============================================================

import h3

H3_RESOLUTION = 8  # ~500m hexagons

bangalore_zones_raw = [
    ("MG Road",                          12.9756, 77.6097, "commercial"),
    ("Brigade Road",                     12.9716, 77.6080, "commercial"),
    ("Residency Road",                   12.9707, 77.6061, "commercial"),
    ("Church Street",                    12.9757, 77.6065, "entertainment"),
    ("Cubbon Park",                      12.9793, 77.5921, "mixed"),
    ("Vidhana Soudha",                   12.9794, 77.5913, "commercial"),
    ("Shivajinagar",                     12.9850, 77.6002, "transit_hub"),
    ("Majestic",                         12.9767, 77.5713, "transit_hub"),
    ("City Market",                      12.9634, 77.5760, "commercial"),
    ("Chickpete",                        12.9605, 77.5757, "commercial"),
    ("Koramangala 1B",                   12.9279, 77.6271, "tech_park"),
    ("Koramangala 4B",                   12.9352, 77.6245, "commercial"),
    ("Koramangala 6B",                   12.9255, 77.6188, "residential"),
    ("Koramangala 8B",                   12.9183, 77.6218, "mixed"),
    ("HSR Layout Sec 1",                 12.9116, 77.6389, "residential"),
    ("HSR Layout Sec 2",                 12.9082, 77.6474, "residential"),
    ("HSR Layout Sec 7",                 12.9047, 77.6387, "tech_park"),
    ("BTM Layout 1",                     12.9166, 77.6101, "residential"),
    ("BTM Layout 2",                     12.9120, 77.6167, "mixed"),
    ("Jayanagar 4B",                     12.9252, 77.5838, "residential"),
    ("Jayanagar 9B",                     12.9171, 77.5836, "residential"),
    ("JP Nagar Phase 1",                 12.9076, 77.5857, "residential"),
    ("JP Nagar Phase 2",                 12.9019, 77.5900, "residential"),
    ("JP Nagar Phase 6",                 12.8938, 77.5934, "residential"),
    ("Banashankari 2nd Stage",           12.9275, 77.5601, "residential"),
    ("Banashankari 3rd Stage",           12.9197, 77.5534, "residential"),
    ("Basavanagudi",                     12.9422, 77.5757, "residential"),
    ("Lalbagh",                          12.9507, 77.5848, "mixed"),
    ("Wilson Garden",                    12.9480, 77.6012, "residential"),
    ("Langford Town",                    12.9547, 77.5978, "residential"),
    ("Hebbal",                           13.0353, 77.5971, "transit_hub"),
    ("Devanahalli",                      13.2473, 77.7113, "transit_hub"),
    ("Yelahanka",                        13.1007, 77.5963, "residential"),
    ("Jakkur",                           13.0693, 77.6099, "residential"),
    ("Sahakara Nagar",                   13.0556, 77.5921, "residential"),
    ("Vidyaranyapura",                   13.0635, 77.5524, "residential"),
    ("Mathikere",                        13.0196, 77.5554, "residential"),
    ("Yeshwanthpur",                     13.0237, 77.5396, "transit_hub"),
    ("Rajajinagar",                      12.9946, 77.5535, "residential"),
    ("Malleswaram",                      13.0035, 77.5673, "residential"),
    ("Sadashivanagar",                   13.0076, 77.5797, "residential"),
    ("RT Nagar",                         13.0207, 77.5973, "residential"),
    ("Thanisandra",                      13.0604, 77.6270, "residential"),
    ("Kogilu",                           13.0762, 77.6199, "residential"),
    ("Bagalur",                          13.1378, 77.6657, "residential"),
    ("Aerospace Park",                   13.2020, 77.7010, "tech_park"),
    ("Whitefield Phase 1",               12.9698, 77.7499, "tech_park"),
    ("Whitefield Phase 2",               12.9750, 77.7550, "tech_park"),
    ("ITPL",                             12.9856, 77.7272, "tech_park"),
    ("Marathahalli",                     12.9591, 77.6974, "commercial"),
    ("Outer Ring Road East",             12.9352, 77.6869, "tech_park"),
    ("Brookefield",                      12.9806, 77.7217, "mixed"),
    ("Varthur",                          12.9388, 77.7480, "residential"),
    ("Kadugodi",                         12.9982, 77.7640, "residential"),
    ("Mahadevapura",                     12.9924, 77.6997, "tech_park"),
    ("KR Puram",                         13.0104, 77.6941, "transit_hub"),
    ("Tin Factory",                      12.9973, 77.6680, "commercial"),
    ("Banaswadi",                        13.0130, 77.6558, "residential"),
    ("CV Raman Nagar",                   12.9962, 77.6604, "residential"),
    ("Indiranagar 100ft Road",           12.9784, 77.6408, "commercial"),
    ("Indiranagar 12th Main",            12.9718, 77.6412, "entertainment"),
    ("HAL Airport Road",                 12.9629, 77.6476, "commercial"),
    ("Domlur",                           12.9610, 77.6384, "mixed"),
    ("Murugeshpalya",                    12.9654, 77.6541, "residential"),
    ("Old Airport Road",                 12.9500, 77.6420, "commercial"),
    ("Kodihalli",                        12.9554, 77.6468, "residential"),
    ("Rajajinagar West",                 12.9901, 77.5481, "residential"),
    ("Basaveshwara Nagar",               12.9946, 77.5361, "residential"),
    ("Nagarbhavi",                       12.9630, 77.5108, "residential"),
    ("Vijayanagar",                      12.9714, 77.5297, "residential"),
    ("Uttarahalli",                      12.8955, 77.5446, "residential"),
    ("Kengeri",                          12.9077, 77.4835, "residential"),
    ("Rajarajeshwari Nagar",             12.9228, 77.5063, "residential"),
    ("Mysore Road",                      12.9418, 77.5266, "transit_hub"),
    ("Chord Road",                       12.9857, 77.5536, "commercial"),
    ("Peenya",                           13.0292, 77.5187, "tech_park"),
    ("Tumkur Road",                      13.0500, 77.5100, "transit_hub"),
    ("Electronic City Phase 1",          12.8399, 77.6770, "tech_park"),
    ("Electronic City Phase 2",          12.8452, 77.6641, "tech_park"),
    ("Infosys Campus EC",                12.8302, 77.6627, "tech_park"),
    ("Wipro Campus EC",                  12.8184, 77.6739, "tech_park"),
    ("Singasandra",                      12.8729, 77.6404, "residential"),
    ("Begur",                            12.8668, 77.6199, "residential"),
    ("Bommanahalli",                     12.8960, 77.6388, "mixed"),
    ("Hongasandra",                      12.8834, 77.6172, "residential"),
    ("Kudlu Gate",                       12.8994, 77.6512, "transit_hub"),
    ("Sarjapur Road KM5",                12.9020, 77.6650, "tech_park"),
    ("Sarjapur Road KM10",               12.8741, 77.6793, "tech_park"),
    ("Sarjapur Road KM15",               12.8450, 77.6950, "tech_park"),
    ("Bellandur",                        12.9255, 77.6716, "tech_park"),
    ("Kadubeesanahalli",                 12.9380, 77.7021, "tech_park"),
    ("Panathur",                         12.9433, 77.7106, "residential"),
    ("Carmelaram",                       12.9087, 77.7263, "residential"),
    ("Kempegowda International Airport", 13.1986, 77.7066, "transit_hub"),
    ("BIAL Road KM10",                   13.1200, 77.6900, "mixed"),
    ("BIAL Road KM20",                   13.1600, 77.6980, "mixed"),
    ("Doddaballapur Road",               13.0800, 77.5700, "mixed"),
    ("Attibele",                         12.7832, 77.7662, "residential"),
    ("Anekal",                           12.7133, 77.6952, "residential"),
    ("Chandapura",                       12.8183, 77.6886, "residential"),
    ("Electronic City Phase 3",          12.8200, 77.6500, "residential"),
    ("Harlur",                           12.9003, 77.6755, "residential"),
    ("Arekere",                          12.8861, 77.6117, "residential"),
    ("Bannerghatta Road KM5",            12.8978, 77.5990, "mixed"),
    ("Bannerghatta Road KM10",           12.8685, 77.5948, "mixed"),
    ("Bannerghatta Road KM15",           12.8392, 77.5907, "mixed"),
    ("Gottigere",                        12.8556, 77.6000, "residential"),
    ("Hulimavu",                         12.8740, 77.6060, "residential"),
    ("Akshayanagar",                     12.8613, 77.6292, "residential"),
    ("Hosa Road",                        12.8770, 77.6698, "residential"),
    ("Kasavanahalli",                    12.9165, 77.7070, "residential"),
    ("Gunjur",                           12.9089, 77.7381, "residential"),
    ("Dommasandra",                      12.8851, 77.7407, "residential"),
    ("Richmond Road",                    12.9628, 77.6009, "commercial"),
    ("Lavelle Road",                     12.9683, 77.5979, "commercial"),
    ("Cunningham Road",                  12.9898, 77.5938, "commercial"),
    ("Palace Road",                      12.9986, 77.5869, "commercial"),
    ("Sankey Road",                      13.0096, 77.5760, "mixed"),
    ("New BEL Road",                     13.0380, 77.5744, "commercial"),
    ("Outer Ring Road North",            13.0450, 77.6180, "mixed"),
    ("Hebbal Flyover",                   13.0450, 77.5850, "transit_hub"),
    ("Nagawara",                         13.0475, 77.6268, "residential"),
    ("Kalyan Nagar",                     13.0298, 77.6474, "residential"),
    ("HBR Layout",                       13.0373, 77.6389, "residential"),
    ("Horamavu",                         13.0330, 77.6624, "residential"),
    ("Ramamurthy Nagar",                 13.0152, 77.6713, "residential"),
    ("Hennur",                           13.0446, 77.6421, "residential"),
    ("Lingarajapuram",                   13.0000, 77.6340, "residential"),
    ("Kammanahalli",                     13.0048, 77.6461, "residential"),
    ("Manyata Tech Park",                13.0465, 77.6213, "tech_park"),
    ("RMZ Infinity",                     12.9876, 77.6954, "tech_park"),
    ("Embassy Tech Village",             12.9258, 77.6946, "tech_park"),
    ("Bagmane Tech Park",                12.9815, 77.6553, "tech_park"),
    ("Global Village Tech Park",         12.9180, 77.5020, "tech_park"),
    ("Software Tech Park Bangalore",     12.9826, 77.5908, "tech_park"),
    ("KSR Bengaluru Station",            12.9767, 77.5713, "transit_hub"),
    ("Yeshwanthpur Junction",            13.0137, 77.5496, "transit_hub"),
    ("Bangalore Cantonment Station",     12.9934, 77.6072, "transit_hub"),
    ("Whitefield Railway Station",       12.9794, 77.7380, "transit_hub"),
    ("Baiyappanahalli Metro",            12.9985, 77.6551, "transit_hub"),
    ("Indiranagar Metro Station",        12.9750, 77.6390, "transit_hub"),
    ("Halasuru Metro Station",           12.9720, 77.6200, "transit_hub"),
    ("Trinity Metro Station",            12.9700, 77.6050, "transit_hub"),
    ("Mantri Mall Malleshwaram",         12.9946, 77.5635, "commercial"),
    ("Orion Mall Rajajinagar",           13.0100, 77.5387, "commercial"),
    ("Phoenix Marketcity Whitefield",    12.9748, 77.7411, "commercial"),
    ("Forum Mall Koramangala",           12.9305, 77.6263, "commercial"),
    ("UB City Mall",                     12.9716, 77.5964, "commercial"),
    ("Garuda Mall",                      12.9720, 77.6080, "commercial"),
    ("Nexus Shantiniketan Mall",         12.9750, 77.7150, "commercial"),
    ("Dasarahalli",                      13.0480, 77.5172, "residential"),
    ("Jalahalli",                        13.0386, 77.5374, "residential"),
    ("HMT Layout",                       13.0436, 77.5454, "residential"),
    ("Bharat Nagar",                     13.0230, 77.5291, "residential"),
    ("Nandini Layout",                   13.0142, 77.5381, "residential"),
    ("Shivanagar",                       12.9842, 77.5221, "residential"),
    ("Kamakshipalya",                    12.9750, 77.5411, "residential"),
    ("Magadi Road",                      12.9620, 77.5344, "transit_hub"),
    ("Hosahalli",                        12.9568, 77.5111, "residential"),
    ("Sunkadakatte",                     12.9542, 77.4966, "residential"),
    ("Manchanabele",                     12.9400, 77.4700, "residential"),
    ("Bidadi",                           12.8017, 77.3929, "residential"),
    ("Nelamangala",                      13.0990, 77.3940, "residential"),
    ("Devanahalli Town",                 13.2449, 77.7226, "residential"),
    ("Hoskote",                          13.0712, 77.7985, "residential"),
    ("Bommasandra",                      12.8106, 77.6812, "residential"),
    ("Jigani",                           12.7909, 77.6372, "residential"),
    ("Anekal Town",                      12.7233, 77.7052, "residential"),
    ("Sarjapur Town",                    12.8657, 77.7858, "residential"),
    ("Kanakapura Road KM10",             12.8800, 77.5680, "mixed"),
    ("Kanakapura Road KM20",             12.8100, 77.5500, "mixed"),
    ("Tumkur Road KM15",                 13.0750, 77.5000, "mixed"),
    ("Mysore Road KM15",                 12.9100, 77.4700, "mixed"),
    ("Hosur Road KM10",                  12.8600, 77.6700, "mixed"),
    ("Old Madras Road KM10",             13.0100, 77.7300, "mixed"),
    ("Bellary Road KM10",                13.0700, 77.6100, "mixed"),
    ("Hennur Road KM5",                  13.0300, 77.6500, "mixed"),
    ("Varthur Road KM5",                 12.9500, 77.7200, "mixed"),
    ("Sarjapur Road KM20",               12.8300, 77.7100, "mixed"),
    ("Bannerghatta Road KM20",           12.8100, 77.5900, "mixed"),
    ("Kanakapura Road KM5",              12.9100, 77.5760, "mixed"),
    ("Hosur Road KM5",                   12.9000, 77.6500, "mixed"),
    ("Old Madras Road KM5",              13.0000, 77.7000, "mixed"),
    ("Bellary Road KM5",                 13.0500, 77.6050, "mixed"),
    ("Nagawara Lake",                    13.0550, 77.6320, "residential"),
    ("Amruthahalli",                     13.0550, 77.5850, "residential"),
    ("Kodigehalli",                      13.0600, 77.5700, "residential"),
    ("Allalasandra",                     13.0650, 77.5980, "residential"),
    ("Byatarayanapura",                  13.0700, 77.5550, "residential"),
    ("Chikkabanaswadi",                  13.0400, 77.6350, "residential"),
    ("Rachenahalli",                     13.0600, 77.6150, "residential"),
    ("Agelakuppam",                      13.0900, 77.6300, "residential"),
    ("Singapura",                        13.0800, 77.5900, "residential"),
    ("Hunsamaranahalli",                 13.1100, 77.5800, "residential"),
    ("Chikkabanavara",                   13.0500, 77.5000, "residential"),
    ("Soladevanahalli",                  13.0700, 77.5200, "residential"),
    ("Abbigere",                         13.0600, 77.4900, "residential"),
    ("Chikkabidarakallu",                13.0400, 77.4800, "residential"),
    ("Nagarur",                          13.1000, 77.5100, "residential"),
    ("Lakshmipura",                      13.0200, 77.5100, "residential"),
    ("Kachohalli",                       13.0300, 77.5000, "residential"),
    ("Haragadde",                        12.7900, 77.7000, "residential"),
    ("Marsur",                           12.7700, 77.7300, "residential"),
    ("Hennagara",                        12.8300, 77.7400, "residential"),
    ("Budigere",                         13.0400, 77.8100, "residential"),
    ("Virgonagar",                       13.0200, 77.7500, "residential"),
    ("Avalahalli",                       13.0000, 77.7700, "residential"),
    ("Samethanahalli",                   12.9800, 77.7800, "residential"),
]

# ── Build Zone DataFrame ─────────────────────────────────────
demand_weight_map = {
    "tech_park":     0.90,
    "transit_hub":   0.85,
    "commercial":    0.80,
    "entertainment": 0.75,
    "mixed":         0.65,
    "residential":   0.45,
}

peak_hour_map = {
    "tech_park":     "08:00-10:00, 17:00-20:00",
    "transit_hub":   "07:00-10:00, 17:00-21:00",
    "commercial":    "10:00-13:00, 17:00-21:00",
    "entertainment": "18:00-23:00",
    "mixed":         "08:00-10:00, 17:00-20:00",
    "residential":   "07:00-09:00, 18:00-21:00",
}

zone_records = []
seen_h3 = {}  # track h3 index duplicates

for idx, (name, lat, lng, z_type) in enumerate(bangalore_zones_raw):
    h3_index = h3.geo_to_h3(lat, lng, H3_RESOLUTION)

    # If H3 cell already used, slightly offset coordinates
    # to generate a neighboring H3 cell
    if h3_index in seen_h3:
        offset = 0.003 * (seen_h3[h3_index] + 1)
        h3_index = h3.geo_to_h3(lat + offset, lng + offset, H3_RESOLUTION)
        seen_h3[h3_index] = seen_h3.get(h3_index, 0) + 1
    else:
        seen_h3[h3_index] = 1

    zone_records.append({
        "zone_id":       f"Z{str(idx+1).zfill(3)}",
        "zone_name":     name,
        "latitude":      lat,
        "longitude":     lng,
        "h3_index":      h3_index,
        "h3_resolution": H3_RESOLUTION,
        "zone_type":     z_type,
        "demand_weight": demand_weight_map[z_type],
        "peak_hours":    peak_hour_map[z_type],
        "city":          CITY,
        "is_active":     True,
    })

zones_df = pd.DataFrame(zone_records)

# ── Save ─────────────────────────────────────────────────────
zones_df.to_csv(os.path.join(RAW_DATA_PATH, "zones.csv"), index=False)

# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("ZONE DATASET - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Zones Generated : {len(zones_df)}")
print(f"  Zone Types            : {zones_df['zone_type'].nunique()}")
print(f"\n  Zone Type Breakdown:")
print(zones_df['zone_type'].value_counts().to_string())
print(f"\n  H3 Resolution         : {H3_RESOLUTION} (~500m hexagons)")
print(f"  Unique H3 Indexes     : {zones_df['h3_index'].nunique()}")
print(f"\n  Sample Records:")
print(zones_df[['zone_id','zone_name','zone_type',
                'demand_weight','h3_index']].head(5).to_string(index=False))
print(f"\n  Saved to: data/raw/zones.csv")
print("\n✓ Block 2 complete - zones.csv generated.")

ZONE DATASET - GENERATION COMPLETE
  Total Zones Generated : 208
  Zone Types            : 6

  Zone Type Breakdown:
zone_type
residential      107
mixed             30
commercial        25
tech_park         24
transit_hub       20
entertainment      2

  H3 Resolution         : 8 (~500m hexagons)
  Unique H3 Indexes     : 206

  Sample Records:
zone_id      zone_name     zone_type  demand_weight        h3_index
   Z001        MG Road    commercial           0.80 8861892e9bfffff
   Z002   Brigade Road    commercial           0.80 88618925a5fffff
   Z003 Residency Road    commercial           0.80 88618925a7fffff
   Z004  Church Street entertainment           0.75 8861892e91fffff
   Z005    Cubbon Park         mixed           0.65 8860145b41fffff

  Saved to: data/raw/zones.csv

✓ Block 2 complete - zones.csv generated.


In [5]:
# ── Trim to exactly 200 zones & fix H3 duplicates ───────────

# Keep first 200 only
zones_df = zones_df.head(200).copy()
zones_df.reset_index(drop=True, inplace=True)

# Reassign zone_ids cleanly
zones_df['zone_id'] = [f"Z{str(i+1).zfill(3)}" for i in range(len(zones_df))]

# Fix any remaining H3 duplicates by forcing unique indexes
seen = {}
fixed_h3 = []
for i, row in zones_df.iterrows():
    h = row['h3_index']
    if h in seen:
        neighbors = h3.k_ring(h, 1)
        neighbors = list(neighbors - set(seen.keys()))
        if neighbors:
            h = neighbors[0]
    seen[h] = 1
    fixed_h3.append(h)

zones_df['h3_index'] = fixed_h3

# Overwrite the saved file
zones_df.to_csv(os.path.join(RAW_DATA_PATH, "zones.csv"), index=False)

# Confirm
print("=" * 60)
print("ZONE DATASET - FINAL CLEAN VERSION")
print("=" * 60)
print(f"  Total Zones     : {len(zones_df)}")
print(f"  Unique H3 Cells : {zones_df['h3_index'].nunique()}")
print(f"\n  Zone Type Breakdown:")
print(zones_df['zone_type'].value_counts().to_string())
print(f"\n  Saved to: data/raw/zones.csv")
print("\n✓ zones.csv finalized - ready for Block 3.")

ZONE DATASET - FINAL CLEAN VERSION
  Total Zones     : 200
  Unique H3 Cells : 200

  Zone Type Breakdown:
zone_type
residential      99
mixed            30
commercial       25
tech_park        24
transit_hub      20
entertainment     2

  Saved to: data/raw/zones.csv

✓ zones.csv finalized - ready for Block 3.


In [6]:
# ============================================================
# BLOCK 3 - Driver Dataset Generation
# 2,000 Rapido drivers across Bangalore zones
# ============================================================

# Vehicle types operating on Rapido Bangalore
VEHICLE_TYPES = ["Bike", "Auto", "EV Bike", "EV Auto"]
VEHICLE_WEIGHTS = [0.45, 0.30, 0.15, 0.10]  # Bike dominant

# Driver status distribution
STATUS_OPTIONS = ["active", "inactive", "suspended"]
STATUS_WEIGHTS = [0.82, 0.13, 0.05]

# Rating distribution - realistic bell curve around 4.2
def generate_rating():
    r = np.random.normal(4.2, 0.4)
    return round(np.clip(r, 1.0, 5.0), 1)

# Experience in months
def generate_experience():
    return int(np.random.exponential(18) + 1)

# Home zone - drivers cluster in residential zones
residential_zones = zones_df[
    zones_df['zone_type'] == 'residential'
]['zone_id'].tolist()

all_zone_ids = zones_df['zone_id'].tolist()

driver_records = []

for i in range(TOTAL_DRIVERS):
    driver_id = f"DRV{str(i+1).zfill(4)}"
    vehicle   = random.choices(VEHICLE_TYPES, VEHICLE_WEIGHTS)[0]
    is_ev     = "EV" in vehicle
    status    = random.choices(STATUS_OPTIONS, STATUS_WEIGHTS)[0]
    rating    = generate_rating()
    exp_months = generate_experience()

    # Home zone - 70% residential, 30% any zone
    if random.random() < 0.70:
        home_zone = random.choice(residential_zones)
    else:
        home_zone = random.choice(all_zone_ids)

    # Current zone - active drivers may be anywhere
    if status == "active":
        current_zone = random.choice(all_zone_ids)
    else:
        current_zone = home_zone

    # Daily target trips - varies by vehicle type
    target_map = {
        "Bike":     random.randint(15, 30),
        "Auto":     random.randint(10, 22),
        "EV Bike":  random.randint(12, 25),
        "EV Auto":  random.randint(8,  18),
    }

    # EV battery capacity in kWh - only relevant for EVs
    battery_capacity = round(random.uniform(2.5, 4.5), 1) if is_ev else None

    # Avg daily earnings in INR - based on vehicle and experience
    base_earnings = {
        "Bike":    600,
        "Auto":    900,
        "EV Bike": 650,
        "EV Auto": 850,
    }
    experience_bonus = min(exp_months * 5, 400)
    avg_daily_earnings = int(
        np.random.normal(
            base_earnings[vehicle] + experience_bonus, 120
        )
    )
    avg_daily_earnings = max(300, avg_daily_earnings)

    # Joining date
    days_back = min(exp_months * 30, 1095)  # max 3 years back
    join_date = START_DATE - timedelta(days=int(days_back))

    # Preferred shift
    shift = random.choices(
        ["morning", "afternoon", "evening", "night", "flexible"],
        weights=[0.25, 0.20, 0.30, 0.10, 0.15]
    )[0]

    driver_records.append({
        "driver_id":           driver_id,
        "vehicle_type":        vehicle,
        "is_ev":               is_ev,
        "status":              status,
        "rating":              rating,
        "experience_months":   exp_months,
        "home_zone_id":        home_zone,
        "current_zone_id":     current_zone,
        "daily_trip_target":   target_map[vehicle],
        "battery_capacity_kwh":battery_capacity,
        "avg_daily_earnings":  avg_daily_earnings,
        "preferred_shift":     shift,
        "join_date":           join_date.date(),
        "city":                CITY,
    })

drivers_df = pd.DataFrame(driver_records)

# ── Save ─────────────────────────────────────────────────────
drivers_df.to_csv(
    os.path.join(RAW_DATA_PATH, "drivers.csv"), index=False
)

# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("DRIVER DATASET - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Drivers         : {len(drivers_df)}")
print(f"  Active Drivers        : {(drivers_df['status']=='active').sum()}")
print(f"  EV Drivers            : {drivers_df['is_ev'].sum()}")
print(f"\n  Vehicle Type Breakdown:")
print(drivers_df['vehicle_type'].value_counts().to_string())
print(f"\n  Status Breakdown:")
print(drivers_df['status'].value_counts().to_string())
print(f"\n  Avg Rating            : {drivers_df['rating'].mean():.2f}")
print(f"  Avg Experience        : {drivers_df['experience_months'].mean():.1f} months")
print(f"  Avg Daily Earnings    : ₹{drivers_df['avg_daily_earnings'].mean():.0f}")
print(f"\n  Preferred Shift:")
print(drivers_df['preferred_shift'].value_counts().to_string())
print(f"\n  Sample Records:")
print(drivers_df[['driver_id','vehicle_type','status',
                  'rating','home_zone_id',
                  'avg_daily_earnings']].head(5).to_string(index=False))
print(f"\n  Saved to: data/raw/drivers.csv")
print("\n✓ Block 3 complete - drivers.csv generated.")

DRIVER DATASET - GENERATION COMPLETE
  Total Drivers         : 2000
  Active Drivers        : 1656
  EV Drivers            : 498

  Vehicle Type Breakdown:
vehicle_type
Bike       890
Auto       612
EV Bike    313
EV Auto    185

  Status Breakdown:
status
active       1656
inactive      256
suspended      88

  Avg Rating            : 4.18
  Avg Experience        : 18.2 months
  Avg Daily Earnings    : ₹812

  Preferred Shift:
preferred_shift
evening      581
morning      514
afternoon    401
flexible     301
night        203

  Sample Records:
driver_id vehicle_type status  rating home_zone_id  avg_daily_earnings
  DRV0001         Auto active     4.4         Z058                1003
  DRV0002         Bike active     3.8         Z129                 658
  DRV0003         Bike active     4.8         Z013                 697
  DRV0004         Bike active     4.0         Z102                 556
  DRV0005         Auto active     4.3         Z097                 725

  Saved to: data/raw/

In [7]:
# ============================================================
# BLOCK 4 - Weather Dataset Generation
# Hourly weather for Bangalore — full year 2024
# 8,760 records (365 days × 24 hours)
# ============================================================

# Bangalore monthly climate profile - realistic
# (avg_temp_c, temp_variation, avg_rain_mm, rainy_day_probability)
monthly_climate = {
    1:  (22.0, 4.0, 0.5,  0.05),   # January  — cool dry
    2:  (25.0, 4.5, 1.0,  0.06),   # February — warming
    3:  (28.0, 5.0, 3.0,  0.10),   # March    — hot
    4:  (30.0, 4.5, 15.0, 0.20),   # April    — pre-monsoon
    5:  (29.0, 4.0, 45.0, 0.35),   # May      — pre-monsoon showers
    6:  (25.0, 3.5, 90.0, 0.60),   # June     — monsoon starts
    7:  (23.0, 3.0, 110.0,0.70),   # July     — peak monsoon
    8:  (23.5, 3.0, 120.0,0.72),   # August   — peak monsoon
    9:  (24.0, 3.5, 150.0,0.75),   # September— heaviest rain
    10: (23.0, 3.5, 120.0,0.65),   # October  — retreating monsoon
    11: (22.0, 4.0, 30.0, 0.25),   # November — post monsoon
    12: (21.0, 4.0, 5.0,  0.08),   # December — cool dry
}

# Bangalore major rain events 2024 - realistic dates
heavy_rain_dates = [
    "2024-04-17", "2024-05-06", "2024-05-19",
    "2024-06-08", "2024-06-22", "2024-07-04",
    "2024-07-18", "2024-08-02", "2024-08-15",
    "2024-08-28", "2024-09-05", "2024-09-12",
    "2024-09-19", "2024-09-26", "2024-10-03",
    "2024-10-15", "2024-10-23", "2024-11-04",
]
heavy_rain_dates = set(heavy_rain_dates)

weather_records = []

current_dt = START_DATE

while current_dt <= datetime(2024, 12, 31, 23, 0, 0):
    month     = current_dt.month
    hour      = current_dt.hour
    date_str  = current_dt.strftime("%Y-%m-%d")

    avg_temp, temp_var, avg_rain, rain_prob = monthly_climate[month]

    # Temperature varies by hour — cooler at night, peak at 2pm
    hour_temp_offset = -3 + 6 * np.sin(np.pi * (hour - 6) / 12)
    temperature = round(
        avg_temp + hour_temp_offset + np.random.normal(0, 1.2), 1
    )

    # Rainfall - heavier rain events on specific dates
    if date_str in heavy_rain_dates:
        is_raining   = True
        rainfall_mm  = round(np.random.uniform(15, 80), 1)
        rain_intensity = (
            "heavy" if rainfall_mm > 40
            else "moderate" if rainfall_mm > 15
            else "light"
        )
    elif random.random() < rain_prob:
        is_raining   = True
        rainfall_mm  = round(np.random.exponential(avg_rain / 10), 1)
        rain_intensity = (
            "heavy" if rainfall_mm > 40
            else "moderate" if rainfall_mm > 15
            else "light"
        )
    else:
        is_raining     = False
        rainfall_mm    = 0.0
        rain_intensity = "none"

    # Visibility - drops during heavy rain
    visibility_map = {
        "none":     round(np.random.uniform(8.0, 12.0), 1),
        "light":    round(np.random.uniform(5.0, 9.0),  1),
        "moderate": round(np.random.uniform(2.0, 5.0),  1),
        "heavy":    round(np.random.uniform(0.5, 2.5),  1),
    }
    visibility_km = visibility_map[rain_intensity]

    # Wind speed
    wind_speed = round(np.random.uniform(5, 25), 1)

    # Humidity
    base_humidity = 60 + (rain_prob * 30)
    humidity = int(np.clip(
        np.random.normal(base_humidity, 10), 30, 99
    ))

    # Weather condition label
    if not is_raining:
        condition = random.choices(
            ["clear", "partly_cloudy", "cloudy"],
            weights=[0.50, 0.30, 0.20]
        )[0]
    else:
        condition = f"rain_{rain_intensity}"

    # Demand multiplier - rain increases ride demand
    demand_multiplier_map = {
        "clear":          round(np.random.uniform(0.90, 1.05), 2),
        "partly_cloudy":  round(np.random.uniform(0.95, 1.05), 2),
        "cloudy":         round(np.random.uniform(1.00, 1.10), 2),
        "rain_light":     round(np.random.uniform(1.15, 1.30), 2),
        "rain_moderate":  round(np.random.uniform(1.30, 1.55), 2),
        "rain_heavy":     round(np.random.uniform(1.55, 2.10), 2),
    }
    demand_multiplier = demand_multiplier_map[condition]

    weather_records.append({
        "timestamp":          current_dt,
        "date":               current_dt.date(),
        "hour":               hour,
        "month":              month,
        "temperature_c":      temperature,
        "humidity_pct":       humidity,
        "wind_speed_kmh":     wind_speed,
        "rainfall_mm":        rainfall_mm,
        "is_raining":         is_raining,
        "rain_intensity":     rain_intensity,
        "visibility_km":      visibility_km,
        "condition":          condition,
        "demand_multiplier":  demand_multiplier,
        "city":               CITY,
    })

    current_dt += timedelta(hours=1)

weather_df = pd.DataFrame(weather_records)

# ── Save ─────────────────────────────────────────────────────
weather_df.to_csv(
    os.path.join(RAW_DATA_PATH, "weather.csv"), index=False
)

# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("WEATHER DATASET - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Records         : {len(weather_df):,}")
print(f"  Date Range            : {weather_df['date'].min()} to {weather_df['date'].max()}")
print(f"  Rainy Hours           : {weather_df['is_raining'].sum():,} ({weather_df['is_raining'].mean()*100:.1f}%)")
print(f"\n  Rain Intensity Breakdown:")
print(weather_df['rain_intensity'].value_counts().to_string())
print(f"\n  Condition Breakdown:")
print(weather_df['condition'].value_counts().to_string())
print(f"\n  Temperature Range     : {weather_df['temperature_c'].min()}°C — {weather_df['temperature_c'].max()}°C")
print(f"  Avg Demand Multiplier : {weather_df['demand_multiplier'].mean():.2f}x")
print(f"  Max Demand Multiplier : {weather_df['demand_multiplier'].max():.2f}x (heavy rain)")
print(f"\n  Sample Records (rainy hours):")
print(weather_df[weather_df['is_raining']][
    ['timestamp','temperature_c','rainfall_mm',
     'rain_intensity','condition','demand_multiplier']
].head(5).to_string(index=False))
print(f"\n  Saved to: data/raw/weather.csv")
print("\n✓ Block 4 complete - weather.csv generated.")

WEATHER DATASET - GENERATION COMPLETE
  Total Records         : 8,784
  Date Range            : 2024-01-01 to 2024-12-31
  Rainy Hours           : 3,512 (40.0%)

  Rain Intensity Breakdown:
rain_intensity
none        5272
light       2422
moderate     763
heavy        327

  Condition Breakdown:
condition
clear            2647
rain_light       2422
partly_cloudy    1550
cloudy           1075
rain_moderate     763
rain_heavy        327

  Temperature Range     : 9.1°C — 35.9°C
  Avg Demand Multiplier : 1.13x
  Max Demand Multiplier : 2.10x (heavy rain)

  Sample Records (rainy hours):
          timestamp  temperature_c  rainfall_mm rain_intensity  condition  demand_multiplier
2024-01-01 09:00:00           22.8          0.0          light rain_light               1.18
2024-01-01 15:00:00           22.7          0.0          light rain_light               1.20
2024-01-01 19:00:00           15.2          0.0          light rain_light               1.22
2024-01-02 05:00:00           17.9   

In [8]:
# ── Fix rainfall_mm inconsistency ───────────────────────────
# Some light rain rows have rainfall_mm = 0.0 - fix them

mask = (weather_df['is_raining'] == True) & (weather_df['rainfall_mm'] == 0.0)
print(f"  Rows to fix: {mask.sum()}")

# Assign small realistic rainfall values
weather_df.loc[
    (weather_df['is_raining'] == True) &
    (weather_df['rainfall_mm'] == 0.0) &
    (weather_df['rain_intensity'] == 'light'),
    'rainfall_mm'
] = np.round(np.random.uniform(0.5, 5.0, mask.sum()), 1)

# Verify fix
remaining = (
    (weather_df['is_raining'] == True) &
    (weather_df['rainfall_mm'] == 0.0)
).sum()

print(f"  Remaining inconsistent rows: {remaining}")

# Overwrite saved file
weather_df.to_csv(
    os.path.join(RAW_DATA_PATH, "weather.csv"), index=False
)

print(f"\n  Avg Demand Multiplier : {weather_df['demand_multiplier'].mean():.2f}x")
print(f"  Total Records         : {len(weather_df):,}")
print("\n✓ Weather fix complete - weather.csv updated.")

  Rows to fix: 80
  Remaining inconsistent rows: 0

  Avg Demand Multiplier : 1.13x
  Total Records         : 8,784

✓ Weather fix complete - weather.csv updated.


In [9]:
# ============================================================
# BLOCK 5 - Events Dataset Generation
# 180 Bangalore events across 2024
# These drive demand spikes in specific zones
# ============================================================

events_raw = [
    # (event_name, date, venue_zone, event_type, expected_attendance, demand_multiplier)

    # ── IPL 2024 - RCB Home Matches at Chinnaswamy ──────────
    ("IPL RCB vs CSK",            "2024-03-25", "MG Road",          "ipl_match",    45000, 2.20),
    ("IPL RCB vs MI",             "2024-03-28", "MG Road",          "ipl_match",    45000, 2.15),
    ("IPL RCB vs KKR",            "2024-04-04", "MG Road",          "ipl_match",    45000, 2.10),
    ("IPL RCB vs RR",             "2024-04-08", "MG Road",          "ipl_match",    45000, 2.18),
    ("IPL RCB vs PBKS",           "2024-04-12", "MG Road",          "ipl_match",    45000, 2.05),
    ("IPL RCB vs SRH",            "2024-04-17", "MG Road",          "ipl_match",    45000, 2.22),
    ("IPL RCB vs DC",             "2024-04-22", "MG Road",          "ipl_match",    45000, 2.08),
    ("IPL RCB vs LSG",            "2024-04-27", "MG Road",          "ipl_match",    45000, 2.12),
    ("IPL RCB vs GT",             "2024-05-04", "MG Road",          "ipl_match",    45000, 2.15),
    ("IPL RCB vs MI Playoff",     "2024-05-18", "MG Road",          "ipl_match",    45000, 2.30),
    ("IPL RCB vs CSK Playoff",    "2024-05-21", "MG Road",          "ipl_match",    45000, 2.35),

    # ── Concerts & Entertainment ─────────────────────────────
    ("Arijit Singh Concert",      "2024-01-14", "Whitefield Phase 1","concert",     25000, 1.85),
    ("AR Rahman Live",            "2024-02-10", "Koramangala 4B",   "concert",      30000, 1.90),
    ("Coldplay World Tour",       "2024-03-15", "Whitefield Phase 1","concert",     50000, 2.40),
    ("Sunburn Festival Day 1",    "2024-12-27", "Electronic City Phase 1","concert",40000, 2.10),
    ("Sunburn Festival Day 2",    "2024-12-28", "Electronic City Phase 1","concert",40000, 2.15),
    ("Sunburn Festival Day 3",    "2024-12-29", "Electronic City Phase 1","concert",35000, 2.00),
    ("NH7 Weekender Day 1",       "2024-11-09", "Koramangala 4B",   "concert",      20000, 1.80),
    ("NH7 Weekender Day 2",       "2024-11-10", "Koramangala 4B",   "concert",      20000, 1.75),
    ("Diljit Dosanjh Concert",    "2024-10-19", "Whitefield Phase 1","concert",     35000, 1.95),
    ("Shreya Ghoshal Live",       "2024-09-21", "MG Road",          "concert",      18000, 1.70),
    ("Ed Sheeran Bangalore",      "2024-03-16", "Whitefield Phase 1","concert",     45000, 2.25),
    ("Prateek Kuhad Live",        "2024-07-06", "Indiranagar 12th Main","concert",  8000, 1.55),
    ("Nucleya Bass Camp",         "2024-08-24", "Koramangala 4B",   "concert",      12000, 1.65),
    ("Shankar Ehsaan Loy Live",   "2024-06-15", "MG Road",          "concert",      15000, 1.68),

    # ── Tech Conferences ─────────────────────────────────────
    ("AWS re:Invent Bangalore",   "2024-01-18", "Whitefield Phase 1","tech_conf",   5000, 1.45),
    ("Google I/O Extended",       "2024-05-15", "Koramangala 1B",   "tech_conf",    3000, 1.35),
    ("Nasscom Tech Summit Day 1", "2024-02-21", "Whitefield Phase 1","tech_conf",  10000, 1.55),
    ("Nasscom Tech Summit Day 2", "2024-02-22", "Whitefield Phase 1","tech_conf",  10000, 1.52),
    ("Nasscom Tech Summit Day 3", "2024-02-23", "Whitefield Phase 1","tech_conf",   8000, 1.48),
    ("BangaloreTech Summit D1",   "2024-09-19", "Whitefield Phase 1","tech_conf",  15000, 1.60),
    ("BangaloreTech Summit D2",   "2024-09-20", "Whitefield Phase 1","tech_conf",  15000, 1.58),
    ("BangaloreTech Summit D3",   "2024-09-21", "Whitefield Phase 1","tech_conf",  12000, 1.55),
    ("Startup India Summit",      "2024-03-08", "Koramangala 1B",   "tech_conf",    4000, 1.38),
    ("Meta Developer Conf",       "2024-07-12", "Indiranagar 100ft Road","tech_conf",2500,1.30),
    ("Microsoft Build Bangalore", "2024-06-20", "Whitefield Phase 1","tech_conf",   4000, 1.40),
    ("Pycon India 2024 Day 1",    "2024-09-20", "ITPL",             "tech_conf",    2000, 1.28),
    ("Pycon India 2024 Day 2",    "2024-09-21", "ITPL",             "tech_conf",    2000, 1.25),
    ("SaaStr Annual Bangalore",   "2024-10-10", "Koramangala 1B",   "tech_conf",    6000, 1.42),
    ("Devoxx India Day 1",        "2024-05-09", "Whitefield Phase 1","tech_conf",   3000, 1.32),
    ("Devoxx India Day 2",        "2024-05-10", "Whitefield Phase 1","tech_conf",   3000, 1.30),

    # ── Festivals & Public Holidays ──────────────────────────
    ("Ugadi Festival",            "2024-04-09", "Basavanagudi",     "festival",     50000, 1.75),
    ("Rama Navami",               "2024-04-17", "Basavanagudi",     "festival",     30000, 1.60),
    ("Ganesh Chaturthi Day 1",    "2024-09-07", "City Market",      "festival",     80000, 1.90),
    ("Ganesh Chaturthi Day 2",    "2024-09-08", "City Market",      "festival",     75000, 1.88),
    ("Ganesh Chaturthi Visarjan", "2024-09-17", "City Market",      "festival",    100000, 2.00),
    ("Navratri Day 1",            "2024-10-03", "Basavanagudi",     "festival",     25000, 1.55),
    ("Navratri Day 9",            "2024-10-12", "Basavanagudi",     "festival",     40000, 1.70),
    ("Dussehra",                  "2024-10-12", "City Market",      "festival",     60000, 1.80),
    ("Diwali Eve",                "2024-11-01", "MG Road",          "festival",     70000, 1.85),
    ("Diwali",                    "2024-11-01", "MG Road",          "festival",     80000, 1.90),
    ("Kannada Rajyotsava",        "2024-11-01", "Vidhana Soudha",   "festival",     50000, 1.65),
    ("Christmas Eve",             "2024-12-24", "MG Road",          "festival",     60000, 1.80),
    ("Christmas Day",             "2024-12-25", "Brigade Road",     "festival",     55000, 1.78),
    ("New Year Eve",              "2024-12-31", "MG Road",          "festival",    120000, 2.50),
    ("Holi",                      "2024-03-25", "Koramangala 4B",   "festival",     30000, 1.60),
    ("Eid ul Fitr",               "2024-04-10", "City Market",      "festival",     40000, 1.65),
    ("Independence Day",          "2024-08-15", "Vidhana Soudha",   "festival",     35000, 1.50),
    ("Republic Day",              "2024-01-26", "Vidhana Soudha",   "festival",     30000, 1.45),
    ("Makara Sankranti",          "2024-01-15", "Lalbagh",          "festival",     25000, 1.40),
    ("Varamahalakshmi",           "2024-08-16", "Basavanagudi",     "festival",     20000, 1.42),

    # ── Bandh & Disruption Events ────────────────────────────
    ("Karnataka Bandh",           "2024-02-26", "Majestic",         "bandh",            0, 0.35),
    ("Bharat Bandh",              "2024-04-10", "Majestic",         "bandh",            0, 0.30),
    ("Auto Strike",               "2024-06-12", "Majestic",         "bandh",            0, 1.80),
    ("Bus Strike BMTC",           "2024-08-05", "Majestic",         "bandh",            0, 1.95),
    ("Metro Disruption North",    "2024-09-14", "Hebbal",           "disruption",       0, 1.65),
    ("Metro Disruption South",    "2024-10-22", "Koramangala 4B",   "disruption",       0, 1.60),
    ("Namma Metro Signal Fail",   "2024-07-30", "MG Road",          "disruption",       0, 1.55),

    # ── Sports Events ────────────────────────────────────────
    ("India vs Aus Test Day 1",   "2024-10-17", "MG Road",          "sports",       35000, 1.75),
    ("India vs Aus Test Day 2",   "2024-10-18", "MG Road",          "sports",       30000, 1.70),
    ("India vs Aus Test Day 3",   "2024-10-19", "MG Road",          "sports",       28000, 1.65),
    ("India vs Eng T20",          "2024-01-17", "MG Road",          "sports",       40000, 1.80),
    ("India vs SA ODI",           "2024-03-02", "MG Road",          "sports",       38000, 1.78),
    ("Bengaluru FC vs Mumbai",    "2024-02-16", "Koramangala 4B",   "sports",       20000, 1.55),
    ("Bengaluru FC vs Goa",       "2024-03-03", "Koramangala 4B",   "sports",       18000, 1.50),
    ("Marathon Bangalore",        "2024-10-20", "MG Road",          "sports",       25000, 1.45),
    ("Cyclothon Bangalore",       "2024-11-03", "MG Road",          "sports",       15000, 1.35),
    ("Pro Kabaddi Bengaluru",     "2024-08-10", "Koramangala 4B",   "sports",       10000, 1.42),

    # ── College & Education Events ───────────────────────────
    ("IIM Bangalore Convocation", "2024-03-30", "Koramangala 1B",   "college_fest", 5000, 1.30),
    ("IISc Tech Fest Day 1",      "2024-02-03", "Malleswaram",      "college_fest", 8000, 1.32),
    ("IISc Tech Fest Day 2",      "2024-02-04", "Malleswaram",      "college_fest", 8000, 1.30),
    ("Christ University Fest",    "2024-02-22", "Koramangala 6B",   "college_fest", 6000, 1.28),
    ("BITS Pilani Bangalore Fest","2024-03-14", "Whitefield Phase 1","college_fest", 5000, 1.25),
    ("PES University Tech Fest",  "2024-09-13", "Outer Ring Road East","college_fest",4000,1.22),
    ("BMS College Fest",          "2024-01-25", "Basavanagudi",     "college_fest", 5000, 1.25),
    ("RV College Cultural",       "2024-02-08", "Jayanagar 4B",     "college_fest", 4000, 1.20),
    ("RVCE Tech Fest",            "2024-03-21", "Jayanagar 4B",     "college_fest", 5000, 1.22),
    ("NMIMS Bangalore MBA Fest",  "2024-08-30", "Koramangala 1B",   "college_fest", 3000, 1.18),

    # ── Shopping & Commercial Events ─────────────────────────
    ("Big Billion Days Bangalore", "2024-10-08","Forum Mall Koramangala","shopping",80000,1.60),
    ("Amazon Great Indian Sale",  "2024-01-13", "Garuda Mall",      "shopping",    60000, 1.45),
    ("Namma Ooru Habba D1",       "2024-12-06", "MG Road",          "shopping",    30000, 1.50),
    ("Namma Ooru Habba D2",       "2024-12-07", "MG Road",          "shopping",    35000, 1.55),
    ("Namma Ooru Habba D3",       "2024-12-08", "Brigade Road",     "shopping",    40000, 1.60),
    ("UB City Art Festival",      "2024-11-15", "UB City Mall",     "shopping",    15000, 1.38),
    ("Lulu Mall Opening",         "2024-07-20", "Hebbal",           "shopping",    50000, 1.70),
    ("Decathlon Fest",            "2024-04-06", "Whitefield Phase 1","shopping",   10000, 1.30),

    # ── Infrastructure Events ────────────────────────────────
    ("Metro Purple Line Extension","2024-06-01","Whitefield Station","infra",           0, 1.25),
    ("Outer Ring Road Blockage",  "2024-03-11", "Outer Ring Road East","infra",         0, 1.40),
    ("Silk Board Junction Work",  "2024-05-20", "HSR Layout Sec 1", "infra",            0, 1.45),
    ("Hebbal Flyover Repair",     "2024-08-12", "Hebbal",           "infra",            0, 1.35),
    ("NICE Road Tollgate Issue",  "2024-09-03", "Electronic City Phase 1","infra",      0, 1.30),
    ("Tumkur Road Widening",      "2024-10-14", "Tumkur Road",      "infra",            0, 1.28),

    # ── Food & Lifestyle Events ──────────────────────────────
    ("Bangalore Food Fest D1",    "2024-01-19", "Indiranagar 12th Main","food_fest",8000,1.40),
    ("Bangalore Food Fest D2",    "2024-01-20", "Indiranagar 12th Main","food_fest",8000,1.38),
    ("Bangalore Food Fest D3",    "2024-01-21", "Indiranagar 12th Main","food_fest",6000,1.35),
    ("Brew & Barbecue Fest",      "2024-02-17", "Koramangala 4B",   "food_fest",    5000, 1.32),
    ("Bangalore Beer Festival",   "2024-03-09", "Indiranagar 12th Main","food_fest",7000,1.38),
    ("Street Food Carnival",      "2024-07-13", "Brigade Road",     "food_fest",    6000, 1.35),
    ("Organic Farmers Market",    "2024-04-14", "Lalbagh",          "food_fest",    3000, 1.18),
    ("Wine & Dine Bangalore",     "2024-11-23", "UB City Mall",     "food_fest",    4000, 1.28),

    # ── New Year & Holiday Specials ──────────────────────────
    ("New Year 2024 Countdown",   "2024-01-01", "MG Road",          "festival",     90000, 2.20),
    ("Valentine Day Bangalore",   "2024-02-14", "Brigade Road",     "festival",     25000, 1.45),
    ("Women Day March",           "2024-03-08", "MG Road",          "festival",     15000, 1.30),
    ("Earth Hour Bangalore",      "2024-03-23", "Cubbon Park",      "festival",      8000, 1.15),
    ("World Music Day",           "2024-06-21", "Indiranagar 12th Main","festival",12000, 1.42),
    ("Raksha Bandhan",            "2024-08-19", "City Market",      "festival",     20000, 1.38),
    ("Onam Celebration",          "2024-09-15", "Koramangala 4B",   "festival",     15000, 1.35),
    ("Durga Puja Bangalore",      "2024-10-11", "Shivajinagar",     "festival",     25000, 1.50),
    ("Guru Nanak Jayanti",        "2024-11-15", "Shivajinagar",     "festival",     18000, 1.35),
    ("Eid ul Adha",               "2024-06-17", "City Market",      "festival",     30000, 1.55),
    ("Good Friday",               "2024-03-29", "Church Street",    "festival",     12000, 1.30),
    ("Easter Sunday",             "2024-03-31", "Church Street",    "festival",     15000, 1.35),

    # ── Government & Political Events ────────────────────────
    ("Karnataka Budget Day",      "2024-02-16", "Vidhana Soudha",   "govt_event",   5000, 1.25),
    ("Lok Sabha Election Day BLR","2024-04-26", "Shivajinagar",     "govt_event",  200000,1.20),
    ("PM Modi Visit Bangalore",   "2024-03-12", "Vidhana Soudha",   "govt_event",  50000, 1.35),
    ("Invest Karnataka Summit D1","2024-02-12","Whitefield Phase 1","govt_event",  10000, 1.40),
    ("Invest Karnataka Summit D2","2024-02-13","Whitefield Phase 1","govt_event",  10000, 1.38),
    ("Aero India 2024 Day 1",     "2024-02-19", "Aerospace Park",   "govt_event",  50000, 1.65),
    ("Aero India 2024 Day 2",     "2024-02-20", "Aerospace Park",   "govt_event",  45000, 1.62),
    ("Aero India 2024 Day 3",     "2024-02-21", "Aerospace Park",   "govt_event",  40000, 1.58),
    ("Aero India 2024 Day 4",     "2024-02-22", "Aerospace Park",   "govt_event",  35000, 1.55),
    ("Aero India 2024 Day 5",     "2024-02-23", "Aerospace Park",   "govt_event",  30000, 1.50),
]

# ── Build Events DataFrame ───────────────────────────────────
event_records = []

for idx, (name, date_str, venue_zone_name,
          event_type, attendance, demand_mult) in enumerate(events_raw):

    # Match venue zone to zone_id
    matched = zones_df[zones_df['zone_name'] == venue_zone_name]
    if len(matched) > 0:
        venue_zone_id  = matched.iloc[0]['zone_id']
        venue_lat      = matched.iloc[0]['latitude']
        venue_lng      = matched.iloc[0]['longitude']
        venue_h3_index = matched.iloc[0]['h3_index']
    else:
        # If zone name not exact match, assign nearest commercial zone
        commercial_zones = zones_df[zones_df['zone_type'] == 'commercial']
        venue_zone_id  = commercial_zones.iloc[0]['zone_id']
        venue_lat      = commercial_zones.iloc[0]['latitude']
        venue_lng      = commercial_zones.iloc[0]['longitude']
        venue_h3_index = commercial_zones.iloc[0]['h3_index']

    event_date = datetime.strptime(date_str, "%Y-%m-%d")

    # Affected radius - how many km around venue sees demand spike
    radius_map = {
        "ipl_match":    3.0,
        "concert":      2.5,
        "tech_conf":    1.5,
        "festival":     4.0,
        "bandh":        15.0,
        "disruption":   5.0,
        "sports":       3.0,
        "college_fest": 1.0,
        "shopping":     2.0,
        "infra":        3.0,
        "food_fest":    1.5,
        "govt_event":   3.0,
    }

    # Duration in hours
    duration_map = {
        "ipl_match":    5,
        "concert":      4,
        "tech_conf":    8,
        "festival":     12,
        "bandh":        12,
        "disruption":   6,
        "sports":       6,
        "college_fest": 10,
        "shopping":     14,
        "infra":        24,
        "food_fest":    8,
        "govt_event":   8,
    }

    # Start hour - when demand spike begins
    start_hour_map = {
        "ipl_match":    18,
        "concert":      19,
        "tech_conf":    8,
        "festival":     6,
        "bandh":        6,
        "disruption":   7,
        "sports":       14,
        "college_fest": 9,
        "shopping":     10,
        "infra":        0,
        "food_fest":    11,
        "govt_event":   9,
    }

    day_of_week  = event_date.strftime("%A")
    is_weekend   = day_of_week in ["Saturday", "Sunday"]
    month        = event_date.month

    event_records.append({
        "event_id":               f"EVT{str(idx+1).zfill(3)}",
        "event_name":             name,
        "event_date":             event_date.date(),
        "day_of_week":            day_of_week,
        "is_weekend":             is_weekend,
        "month":                  month,
        "venue_zone_id":          venue_zone_id,
        "venue_zone_name":        venue_zone_name,
        "venue_lat":              venue_lat,
        "venue_lng":              venue_lng,
        "venue_h3_index":         venue_h3_index,
        "event_type":             event_type,
        "expected_attendance":    attendance,
        "demand_multiplier":      demand_mult,
        "affected_radius_km":     radius_map[event_type],
        "duration_hours":         duration_map[event_type],
        "start_hour":             start_hour_map[event_type],
        "city":                   CITY,
    })

events_df = pd.DataFrame(event_records)

# ── Save ─────────────────────────────────────────────────────
events_df.to_csv(
    os.path.join(RAW_DATA_PATH, "events.csv"), index=False
)

# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("EVENTS DATASET - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Events          : {len(events_df)}")
print(f"  Date Range            : {events_df['event_date'].min()} to {events_df['event_date'].max()}")
print(f"  Weekend Events        : {events_df['is_weekend'].sum()}")
print(f"\n  Event Type Breakdown:")
print(events_df['event_type'].value_counts().to_string())
print(f"\n  Top 5 Highest Demand Events:")
print(events_df.nlargest(5, 'demand_multiplier')[
    ['event_name','event_date','event_type','demand_multiplier']
].to_string(index=False))
print(f"\n  Avg Demand Multiplier : {events_df['demand_multiplier'].mean():.2f}x")
print(f"  Max Demand Multiplier : {events_df['demand_multiplier'].max():.2f}x")
print(f"\n  Saved to: data/raw/events.csv")
print("\n✓ Block 5 complete - events.csv generated.")

EVENTS DATASET - GENERATION COMPLETE
  Total Events          : 132
  Date Range            : 2024-01-01 to 2024-12-31
  Weekend Events        : 47

  Event Type Breakdown:
event_type
festival        32
tech_conf       16
concert         14
ipl_match       11
sports          10
college_fest    10
govt_event      10
shopping         8
food_fest        8
infra            6
bandh            4
disruption       3

  Top 5 Highest Demand Events:
            event_name event_date event_type  demand_multiplier
          New Year Eve 2024-12-31   festival               2.50
   Coldplay World Tour 2024-03-15    concert               2.40
IPL RCB vs CSK Playoff 2024-05-21  ipl_match               2.35
 IPL RCB vs MI Playoff 2024-05-18  ipl_match               2.30
  Ed Sheeran Bangalore 2024-03-16    concert               2.25

  Avg Demand Multiplier : 1.58x
  Max Demand Multiplier : 2.50x

  Saved to: data/raw/events.csv

✓ Block 5 complete - events.csv generated.


In [10]:
# ============================================================
# BLOCK 6 - EV Charging Stations Dataset Generation
# 85 charging stations across Bangalore
# ============================================================

# Real charging networks operating in Bangalore
charging_networks = ["Tata Power EV", "Ather Grid", "Statiq",
                     "ChargeZone", "BESCOM Public", "Zeon Charging",
                     "Rapido Internal", "BPCL Pulse"]

network_weights = [0.20, 0.18, 0.15, 0.12, 0.12, 0.10, 0.08, 0.05]

# Charger types
charger_types = ["slow_ac", "fast_dc", "ultra_fast_dc"]
charger_weights = [0.45, 0.40, 0.15]

# Zones that realistically have charging stations
charging_zone_types = ["tech_park", "transit_hub", "commercial", "mixed"]
charging_zones = zones_df[
    zones_df['zone_type'].isin(charging_zone_types)
]['zone_id'].tolist()

# Add some residential zones too - home charging points
residential_charging = zones_df[
    zones_df['zone_type'] == 'residential'
].sample(20, random_state=42)['zone_id'].tolist()

all_charging_zones = charging_zones + residential_charging

station_records = []

for i in range(85):
    station_id = f"CS{str(i+1).zfill(3)}"
    network    = random.choices(charging_networks, network_weights)[0]
    zone_id    = random.choice(all_charging_zones)

    # Get zone details
    zone_info  = zones_df[zones_df['zone_id'] == zone_id].iloc[0]
    zone_type  = zone_info['zone_type']

    # Charger type - tech parks and transit hubs get faster chargers
    if zone_type in ["tech_park", "transit_hub"]:
        charger_type = random.choices(
            charger_types, [0.20, 0.55, 0.25]
        )[0]
    elif zone_type == "residential":
        charger_type = random.choices(
            charger_types, [0.80, 0.18, 0.02]
        )[0]
    else:
        charger_type = random.choices(
            charger_types, charger_weights
        )[0]

    # Number of charging ports
    port_map = {
        "slow_ac":       random.randint(2, 8),
        "fast_dc":       random.randint(2, 6),
        "ultra_fast_dc": random.randint(1, 4),
    }
    total_ports = port_map[charger_type]

    # Power output in kW
    power_map = {
        "slow_ac":       random.choice([3.3, 7.2, 11.0]),
        "fast_dc":       random.choice([30.0, 50.0, 60.0]),
        "ultra_fast_dc": random.choice([100.0, 150.0, 240.0]),
    }
    power_kw = power_map[charger_type]

    # Charging cost per unit (INR/kWh)
    cost_map = {
        "slow_ac":       round(random.uniform(6.0, 9.0), 1),
        "fast_dc":       round(random.uniform(12.0, 18.0), 1),
        "ultra_fast_dc": round(random.uniform(18.0, 25.0), 1),
    }
    cost_per_kwh = cost_map[charger_type]

    # Operating hours
    if zone_type in ["tech_park", "commercial"]:
        operating_hours = "06:00-23:00"
        is_24hr = False
    elif zone_type == "transit_hub":
        operating_hours = "24x7"
        is_24hr = True
    else:
        operating_hours = "07:00-22:00"
        is_24hr = False

    # Average utilization rate
    utilization_base = {
        "tech_park":   0.72,
        "transit_hub": 0.68,
        "commercial":  0.55,
        "mixed":       0.50,
        "residential": 0.35,
    }
    avg_utilization = round(
        np.clip(
            np.random.normal(utilization_base[zone_type], 0.10),
            0.10, 0.95
        ), 2
    )

    # Monthly revenue estimate
    # Revenue = ports × power_kw × daily_hours × utilization × cost × 30
    daily_hours = 17 if not is_24hr else 24
    monthly_revenue = int(
        total_ports * power_kw *
        daily_hours * avg_utilization *
        cost_per_kwh * 30
    )

    # Installation year
    install_year = random.choices(
        [2021, 2022, 2023, 2024],
        weights=[0.10, 0.20, 0.40, 0.30]
    )[0]

    # Is operational
    is_operational = random.choices(
        [True, False], weights=[0.88, 0.12]
    )[0]

    # Slight coordinate offset from zone center
    lat_offset = np.random.uniform(-0.005, 0.005)
    lng_offset = np.random.uniform(-0.005, 0.005)

    station_records.append({
        "station_id":        station_id,
        "network":           network,
        "zone_id":           zone_id,
        "zone_name":         zone_info['zone_name'],
        "zone_type":         zone_type,
        "latitude":          round(zone_info['latitude'] + lat_offset, 6),
        "longitude":         round(zone_info['longitude'] + lng_offset, 6),
        "h3_index":          zone_info['h3_index'],
        "charger_type":      charger_type,
        "total_ports":       total_ports,
        "power_kw":          power_kw,
        "cost_per_kwh":      cost_per_kwh,
        "operating_hours":   operating_hours,
        "is_24hr":           is_24hr,
        "avg_utilization":   avg_utilization,
        "monthly_revenue_inr": monthly_revenue,
        "install_year":      install_year,
        "is_operational":    is_operational,
        "city":              CITY,
    })

stations_df = pd.DataFrame(station_records)

# ── Save ─────────────────────────────────────────────────────
stations_df.to_csv(
    os.path.join(RAW_DATA_PATH, "charging_stations.csv"),
    index=False
)

# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("CHARGING STATIONS - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Stations        : {len(stations_df)}")
print(f"  Operational           : {stations_df['is_operational'].sum()}")
print(f"\n  Network Breakdown:")
print(stations_df['network'].value_counts().to_string())
print(f"\n  Charger Type Breakdown:")
print(stations_df['charger_type'].value_counts().to_string())
print(f"\n  Zone Type Coverage:")
print(stations_df['zone_type'].value_counts().to_string())
print(f"\n  Avg Utilization       : {stations_df['avg_utilization'].mean():.1%}")
print(f"  Avg Monthly Revenue   : ₹{stations_df['monthly_revenue_inr'].mean():,.0f}")
print(f"  Total Monthly Revenue : ₹{stations_df['monthly_revenue_inr'].sum():,.0f}")
print(f"\n  Sample Records:")
print(stations_df[[
    'station_id','network','zone_name',
    'charger_type','total_ports',
    'power_kw','avg_utilization'
]].head(5).to_string(index=False))
print(f"\n  Saved to: data/raw/charging_stations.csv")
print("\n✓ Block 6 complete - charging_stations.csv generated.")

CHARGING STATIONS - GENERATION COMPLETE
  Total Stations        : 85
  Operational           : 76

  Network Breakdown:
network
Ather Grid         19
Statiq             14
Tata Power EV      12
Zeon Charging      11
BESCOM Public      10
Rapido Internal     7
ChargeZone          7
BPCL Pulse          5

  Charger Type Breakdown:
charger_type
slow_ac          42
fast_dc          28
ultra_fast_dc    15

  Zone Type Coverage:
zone_type
tech_park      22
mixed          20
commercial     19
residential    16
transit_hub     8

  Avg Utilization       : 56.2%
  Avg Monthly Revenue   : ₹862,423
  Total Monthly Revenue : ₹73,305,927

  Sample Records:
station_id         network              zone_name  charger_type  total_ports  power_kw  avg_utilization
     CS001      BPCL Pulse Indiranagar 100ft Road       fast_dc            4      30.0             0.54
     CS002 Rapido Internal        Cunningham Road       slow_ac            5       7.2             0.43
     CS003          Statiq          

In [11]:
# ============================================================
# BLOCK 7 - Rides Dataset Generation
# 500,000 ride records - the core dataset
# Incorporates weather + events + zone demand weights
# ============================================================

from datetime import date

# ── Pre-build lookup structures for speed ────────────────────

# Zone lookup dict - faster than DataFrame query in loop
zone_dict = zones_df.set_index('zone_id').to_dict('index')
zone_ids  = zones_df['zone_id'].tolist()

# Active driver ids
active_drivers = drivers_df[
    drivers_df['status'] == 'active'
]['driver_id'].tolist()

# Weather lookup - keyed by (date_str, hour)
weather_df['date_str'] = weather_df['date'].astype(str)
weather_lookup = {}
for _, row in weather_df.iterrows():
    weather_lookup[(row['date_str'], row['hour'])] = {
        'condition':         row['condition'],
        'rainfall_mm':       row['rainfall_mm'],
        'is_raining':        row['is_raining'],
        'demand_multiplier': row['demand_multiplier'],
        'temperature_c':     row['temperature_c'],
    }

# Event lookup - keyed by date_str
events_df['date_str'] = events_df['event_date'].astype(str)
event_lookup = {}
for _, row in events_df.iterrows():
    d = row['date_str']
    if d not in event_lookup:
        event_lookup[d] = []
    event_lookup[d].append({
        'event_id':          row['event_id'],
        'event_name':        row['event_name'],
        'event_type':        row['event_type'],
        'venue_zone_id':     row['venue_zone_id'],
        'demand_multiplier': row['demand_multiplier'],
        'start_hour':        row['start_hour'],
        'duration_hours':    row['duration_hours'],
        'affected_radius_km':row['affected_radius_km'],
    })

# Vehicle type by driver
driver_vehicle = drivers_df.set_index(
    'driver_id'
)['vehicle_type'].to_dict()

# Fare base rates by vehicle type (INR per km)
fare_base = {
    "Bike":     12,
    "Auto":     18,
    "EV Bike":  13,
    "EV Auto":  17,
}

# Ride status distribution
ride_statuses   = ["completed", "cancelled_by_rider",
                   "cancelled_by_driver", "no_driver_found"]
ride_status_wts = [0.78, 0.10, 0.07, 0.05]

# Payment modes
payment_modes   = ["UPI", "Cash", "Wallet", "Card"]
payment_wts     = [0.52, 0.28, 0.12, 0.08]

# Cancellation reasons
cancel_rider_reasons = [
    "took_auto_instead", "wait_too_long",
    "wrong_pickup", "changed_plans", "price_too_high"
]
cancel_driver_reasons = [
    "rider_too_far", "low_fare", "area_not_preferred",
    "vehicle_issue", "personal_reason"
]

# Zone demand weights for weighted sampling
zone_demand_weights = zones_df['demand_weight'].tolist()

print("Lookup structures built. Starting ride generation...")
print(f"Generating {TOTAL_RIDES:,} rides — please wait 2-3 minutes...\n")

# ── Generate Rides ───────────────────────────────────────────
ride_records = []

# Pre-generate all timestamps for speed
all_dates = pd.date_range(START_DATE, END_DATE, freq='D')

rides_per_day_base = TOTAL_RIDES // len(all_dates)  # ~1,366 per day

ride_counter = 0

for current_date in all_dates:
    date_str    = current_date.strftime("%Y-%m-%d")
    day_of_week = current_date.strftime("%A")
    is_weekend  = day_of_week in ["Saturday", "Sunday"]
    month       = current_date.month

    # Weekend has more rides
    daily_rides = int(rides_per_day_base * (
        1.15 if is_weekend else 1.0
    ))

    # Events on this date boost daily rides
    day_events = event_lookup.get(date_str, [])
    if day_events:
        max_event_mult = max(e['demand_multiplier'] for e in day_events)
        daily_rides = int(daily_rides * min(max_event_mult, 1.5))

    for _ in range(daily_rides):
        if ride_counter >= TOTAL_RIDES:
            break

        # Random hour - weighted by realistic demand pattern
        # Peak: 8-10am, 5-8pm. Low: 2-5am
        hour_weights = [
            0.8, 0.5, 0.3, 0.2, 0.2, 0.5,   # 0-5am
            1.5, 2.5, 3.5, 3.0, 2.5, 2.0,   # 6-11am
            2.5, 2.0, 1.8, 1.5, 1.8, 3.5,   # 12-5pm
            4.0, 3.5, 3.0, 2.5, 2.0, 1.5,   # 6-11pm
        ]
        hour   = random.choices(range(24), weights=hour_weights)[0]
        minute = random.randint(0, 59)
        second = random.randint(0, 59)

        ride_dt = current_date + timedelta(
            hours=hour, minutes=minute, seconds=second
        )

        # Weather at this hour
        weather = weather_lookup.get(
            (date_str, hour),
            {'condition':'clear','rainfall_mm':0,
             'is_raining':False,'demand_multiplier':1.0,
             'temperature_c':25.0}
        )

        # Pickup zone - weighted by demand
        pickup_zone_id = random.choices(
            zone_ids, weights=zone_demand_weights
        )[0]
        pickup_info    = zone_dict[pickup_zone_id]

        # Dropoff zone - different from pickup
        dropoff_zone_id = pickup_zone_id
        while dropoff_zone_id == pickup_zone_id:
            dropoff_zone_id = random.choices(
                zone_ids, weights=zone_demand_weights
            )[0]
        dropoff_info = zone_dict[dropoff_zone_id]

        # Distance - based on zone coordinate difference
        lat_diff = abs(
            pickup_info['latitude'] - dropoff_info['latitude']
        )
        lng_diff = abs(
            pickup_info['longitude'] - dropoff_info['longitude']
        )
        base_dist = (lat_diff + lng_diff) * 111  # rough km conversion
        distance_km = round(
            max(0.5, min(base_dist * random.uniform(0.8, 1.4), 45.0)),
            1
        )

        # Assign driver
        driver_id    = random.choice(active_drivers)
        vehicle_type = driver_vehicle.get(driver_id, "Bike")

        # Ride status
        status = random.choices(ride_statuses, ride_status_wts)[0]

        # Fare calculation
        base_fare    = fare_base.get(vehicle_type, 12)
        fare_amount  = round(
            max(15, base_fare * distance_km +
                random.uniform(-5, 15)), 0
        )

        # Surge pricing - rain or events
        surge_mult = 1.0
        if weather['is_raining']:
            surge_mult *= weather['demand_multiplier']
        if day_events:
            for evt in day_events:
                if (evt['start_hour'] <= hour <=
                        evt['start_hour'] + evt['duration_hours']):
                    if evt['venue_zone_id'] == pickup_zone_id:
                        surge_mult *= min(evt['demand_multiplier'], 1.8)
                        break

        surge_mult  = round(min(surge_mult, 2.5), 2)
        fare_amount = round(fare_amount * surge_mult, 0)

        # Duration — based on distance + traffic
        traffic_mult = random.uniform(1.0, 2.2)
        if hour in range(8, 11) or hour in range(17, 21):
            traffic_mult *= 1.4  # peak hour traffic
        duration_min = round(
            (distance_km / 20) * 60 * traffic_mult, 0
        )  # 20kmph avg speed in city

        # Wait time - longer in dead zones
        wait_time_min = round(random.uniform(1, 18), 0)

        # Rating - only for completed rides
        if status == "completed":
            ride_rating = round(
                np.clip(np.random.normal(4.1, 0.6), 1.0, 5.0), 1
            )
            payment_mode = random.choices(
                payment_modes, payment_wts
            )[0]
            cancellation_reason = None
        elif status == "cancelled_by_rider":
            ride_rating         = None
            payment_mode        = None
            cancellation_reason = random.choice(cancel_rider_reasons)
            fare_amount         = 0
        elif status == "cancelled_by_driver":
            ride_rating         = None
            payment_mode        = None
            cancellation_reason = random.choice(cancel_driver_reasons)
            fare_amount         = 0
        else:  # no_driver_found
            ride_rating         = None
            payment_mode        = None
            cancellation_reason = "no_driver_available"
            fare_amount         = 0
            wait_time_min       = 20

        # Event context
        active_event_id   = None
        active_event_type = None
        if day_events:
            for evt in day_events:
                if (evt['start_hour'] <= hour <=
                        evt['start_hour'] + evt['duration_hours']):
                    active_event_id   = evt['event_id']
                    active_event_type = evt['event_type']
                    break

        ride_records.append({
            "ride_id":              f"R{str(ride_counter+1).zfill(7)}",
            "ride_datetime":        ride_dt,
            "date":                 current_date.date(),
            "hour":                 hour,
            "day_of_week":          day_of_week,
            "is_weekend":           is_weekend,
            "month":                month,
            "pickup_zone_id":       pickup_zone_id,
            "pickup_zone_name":     pickup_info['zone_name'],
            "pickup_zone_type":     pickup_info['zone_type'],
            "pickup_h3_index":      pickup_info['h3_index'],
            "dropoff_zone_id":      dropoff_zone_id,
            "dropoff_zone_name":    dropoff_info['zone_name'],
            "dropoff_zone_type":    dropoff_info['zone_type'],
            "dropoff_h3_index":     dropoff_info['h3_index'],
            "driver_id":            driver_id,
            "vehicle_type":         vehicle_type,
            "distance_km":          distance_km,
            "duration_min":         duration_min,
            "wait_time_min":        wait_time_min,
            "fare_amount":          fare_amount,
            "surge_multiplier":     surge_mult,
            "status":               status,
            "payment_mode":         payment_mode,
            "ride_rating":          ride_rating,
            "cancellation_reason":  cancellation_reason,
            "weather_condition":    weather['condition'],
            "rainfall_mm":          weather['rainfall_mm'],
            "is_raining":           weather['is_raining'],
            "temperature_c":        weather['temperature_c'],
            "active_event_id":      active_event_id,
            "active_event_type":    active_event_type,
            "city":                 CITY,
        })

        ride_counter += 1

    if ride_counter >= TOTAL_RIDES:
        break

print(f"  Rides generated: {ride_counter:,}")
print("  Building DataFrame...")

rides_df = pd.DataFrame(ride_records)

# ── Save ─────────────────────────────────────────────────────
print("  Saving to CSV - this may take 30-60 seconds...")
rides_df.to_csv(
    os.path.join(RAW_DATA_PATH, "rides.csv"), index=False
)

# ── Validation ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("RIDES DATASET - GENERATION COMPLETE")
print("=" * 60)
print(f"  Total Rides           : {len(rides_df):,}")
print(f"  Date Range            : {rides_df['date'].min()} to {rides_df['date'].max()}")
print(f"\n  Status Breakdown:")
print(rides_df['status'].value_counts().to_string())
print(f"\n  Vehicle Type Breakdown:")
print(rides_df['vehicle_type'].value_counts().to_string())
print(f"\n  Payment Mode Breakdown:")
print(rides_df[rides_df['payment_mode'].notna()]['payment_mode'].value_counts().to_string())
print(f"\n  Avg Fare (completed) : ₹{rides_df[rides_df['status']=='completed']['fare_amount'].mean():.0f}")
print(f"  Avg Distance         : {rides_df['distance_km'].mean():.1f} km")
print(f"  Avg Wait Time        : {rides_df['wait_time_min'].mean():.1f} mins")
print(f"  Avg Rating           : {rides_df['ride_rating'].mean():.2f}")
print(f"  Surge Rides          : {(rides_df['surge_multiplier']>1.0).sum():,} ({(rides_df['surge_multiplier']>1.0).mean()*100:.1f}%)")
print(f"  Rainy Rides          : {rides_df['is_raining'].sum():,} ({rides_df['is_raining'].mean()*100:.1f}%)")
print(f"  Event-linked Rides   : {rides_df['active_event_id'].notna().sum():,}")
print(f"\n  Top 5 Pickup Zones:")
print(rides_df['pickup_zone_name'].value_counts().head(5).to_string())
print(f"\n  Saved to: data/raw/rides.csv")
print("\n✓ Block 7 complete - rides.csv generated.")

Lookup structures built. Starting ride generation...
Generating 500,000 rides — please wait 2-3 minutes...

  Rides generated: 500,000
  Building DataFrame...
  Saving to CSV - this may take 30-60 seconds...

RIDES DATASET - GENERATION COMPLETE
  Total Rides           : 500,000
  Date Range            : 2024-01-01 to 2024-11-04

  Status Breakdown:
status
completed              389828
cancelled_by_rider      49949
cancelled_by_driver     35091
no_driver_found         25132

  Vehicle Type Breakdown:
vehicle_type
Bike       223119
Auto       153840
EV Bike     76948
EV Auto     46093

  Payment Mode Breakdown:
payment_mode
UPI       202610
Cash      109143
Wallet     46835
Card       31240

  Avg Fare (completed) : ₹344
  Avg Distance         : 20.4 km
  Avg Wait Time        : 10.0 mins
  Avg Rating           : 4.08
  Surge Rides          : 218,968 (43.8%)
  Rainy Rides          : 218,510 (43.7%)
  Event-linked Rides   : 113,806

  Top 5 Pickup Zones:
pickup_zone_name
HSR Layout Sec 7  

In [12]:
# ============================================================
# FINAL BLOCK — Complete Dataset Validation Report
# ============================================================

import os

datasets = {
    "zones.csv":             (zones_df,    200),
    "drivers.csv":           (drivers_df,  2000),
    "weather.csv":           (weather_df,  8784),
    "events.csv":            (events_df,   132),
    "charging_stations.csv": (stations_df, 85),
    "rides.csv":             (rides_df,    500000),
}

print("=" * 60)
print("  URBAN MOBILITY INTELLIGENCE SYSTEM")
print("  DATASET GENERATION — COMPLETE VALIDATION REPORT")
print("=" * 60)

all_passed = True
total_rows = 0

for filename, (df, expected_rows) in datasets.items():
    filepath  = os.path.join(RAW_DATA_PATH, filename)
    exists    = os.path.exists(filepath)
    actual    = len(df)
    size_mb   = os.path.getsize(filepath) / (1024*1024) if exists else 0
    status    = "✓ PASS" if exists else "✗ FAIL"
    total_rows += actual

    if not exists:
        all_passed = False

    print(f"\n  {status} — {filename}")
    print(f"         Rows     : {actual:>10,}")
    print(f"         Columns  : {len(df.columns):>10}")
    print(f"         File Size: {size_mb:>9.1f} MB")

print("\n" + "=" * 60)
print(f"  TOTAL ROWS ACROSS ALL DATASETS : {total_rows:,}")
print(f"  ALL FILES SAVED SUCCESSFULLY   : {all_passed}")
print("=" * 60)

print("""
  DATASET OVERVIEW
  ─────────────────────────────────────────────────────
  City Simulated    : Bangalore, Karnataka
  Company Simulated : Rapido
  Simulation Period : Jan 2024 — Nov 2024

  zones.csv              200 zones   | Real Bangalore neighborhoods
  drivers.csv          2,000 drivers | EV + petrol, ratings, shifts
  weather.csv          8,784 hours   | Monsoon-accurate, demand multipliers
  events.csv             132 events  | IPL, concerts, bandhs, festivals
  charging_stations.csv   85 stations | 8 real networks, 3 charger types
  rides.csv          500,000 rides   | Weather + event + surge integrated
  ─────────────────────────────────────────────────────
""")

print("  KEY BUSINESS METRICS FROM DATA")
print("  ─────────────────────────────────────────────────────")

completed = rides_df[rides_df['status'] == 'completed']
cancelled = rides_df[rides_df['status'].isin(
    ['cancelled_by_rider','cancelled_by_driver']
)]

total_revenue    = completed['fare_amount'].sum()
avg_fare         = completed['fare_amount'].mean()
completion_rate  = len(completed) / len(rides_df) * 100
cancellation_rate = len(cancelled) / len(rides_df) * 100
surge_rate       = (rides_df['surge_multiplier'] > 1.0).mean() * 100
ev_share         = rides_df['vehicle_type'].str.contains('EV').mean() * 100
upi_share        = (completed['payment_mode'] == 'UPI').mean() * 100
avg_rating       = rides_df['ride_rating'].mean()
event_rides_pct  = rides_df['active_event_id'].notna().mean() * 100
rainy_rides_pct  = rides_df['is_raining'].mean() * 100

print(f"  Total Revenue Generated  : ₹{total_revenue:>15,.0f}")
print(f"  Avg Fare per Ride        : ₹{avg_fare:>15,.0f}")
print(f"  Completion Rate          :  {completion_rate:>14.1f}%")
print(f"  Cancellation Rate        :  {cancellation_rate:>14.1f}%")
print(f"  Surge Pricing Rate       :  {surge_rate:>14.1f}%")
print(f"  EV Ride Share            :  {ev_share:>14.1f}%")
print(f"  UPI Payment Share        :  {upi_share:>14.1f}%")
print(f"  Avg Ride Rating          :  {avg_rating:>14.2f} / 5.0")
print(f"  Event-driven Rides       :  {event_rides_pct:>14.1f}%")
print(f"  Rainy Weather Rides      :  {rainy_rides_pct:>14.1f}%")

print("\n  ─────────────────────────────────────────────────────")
print("  TOP 5 PICKUP ZONES (by ride volume)")
print("  ─────────────────────────────────────────────────────")
top_zones = rides_df['pickup_zone_name'].value_counts().head(5)
for zone, count in top_zones.items():
    pct = count / len(rides_df) * 100
    print(f"  {zone:<35} {count:>6,} rides ({pct:.1f}%)")

print("\n  ─────────────────────────────────────────────────────")
print("  PEAK HOURS BY RIDE VOLUME")
print("  ─────────────────────────────────────────────────────")
peak_hours = rides_df.groupby('hour').size().nlargest(5)
for hr, count in peak_hours.items():
    print(f"  {hr:02d}:00  —  {count:>7,} rides")

print("\n  ─────────────────────────────────────────────────────")
print("  EV FLEET SUMMARY")
print("  ─────────────────────────────────────────────────────")
ev_drivers_count    = drivers_df['is_ev'].sum()
ev_stations_active  = stations_df['is_operational'].sum()
ev_monthly_rev      = stations_df['monthly_revenue_inr'].sum()
print(f"  EV Drivers           : {ev_drivers_count}")
print(f"  Charging Stations    : {ev_stations_active} operational")
print(f"  Monthly Charging Rev : ₹{ev_monthly_rev:,.0f}")

print("\n" + "=" * 60)
print("  ✓ NOTEBOOK 01 COMPLETE — ALL 6 DATASETS GENERATED")
print("  Next  →  Notebook 02 — PostgreSQL Database Loading")
print("=" * 60)

  URBAN MOBILITY INTELLIGENCE SYSTEM
  DATASET GENERATION — COMPLETE VALIDATION REPORT

  ✓ PASS — zones.csv
         Rows     :        200
         Columns  :         11
         File Size:       0.0 MB

  ✓ PASS — drivers.csv
         Rows     :      2,000
         Columns  :         14
         File Size:       0.2 MB

  ✓ PASS — weather.csv
         Rows     :      8,784
         Columns  :         15
         File Size:       0.8 MB

  ✓ PASS — events.csv
         Rows     :        132
         Columns  :         19
         File Size:       0.0 MB

  ✓ PASS — charging_stations.csv
         Rows     :         85
         Columns  :         19
         File Size:       0.0 MB

  ✓ PASS — rides.csv
         Rows     :    500,000
         Columns  :         33
         File Size:     121.7 MB

  TOTAL ROWS ACROSS ALL DATASETS : 511,201
  ALL FILES SAVED SUCCESSFULLY   : True

  DATASET OVERVIEW
  ─────────────────────────────────────────────────────
  City Simulated    : Bangalore, K